# ConceptNet-100k → recursive graph dictionary v2 — bounded CPU pipeline / L4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SemanticMap/semgraphex/blob/feature/recursive-lossless-graph-dictionary-v2/notebooks/15_conceptnet_100k_l4_bounded_cpu_pipeline_colab.ipynb)

Отдельный воспроизводимый эксперимент PR #13 для нового **bounded extraction + VF2 pipeline**. Один общий CPU budget делится между потоками извлечения ego-графов и spawn-процессами exact VF2 matching; суммарное число рабочих исполнителей не превышает `CPU_WORKERS`.

Используются те же ConceptNet-100k данные, тот же recursive lossless codec, Drive checkpoints и final→level0 exact decode, что и в notebook 14. Отличается только CPU execution model полного census.

**Runtime:** Colab L4 GPU + High-RAM. При обрыве используйте `RUN_MODE='RESUME'` с неизменными `PINNED_SHA`, входом, `RUN_NAME`, `CPU_WORKERS` и устройством.

In [ ]:
# ===== USER PARAMETERS =====
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
BRANCH = 'feature/recursive-lossless-graph-dictionary-v2'
PINNED_SHA = '779ecd2c24644d6edf7b6c827ea061a8f0b63138'  # Код эксперимента; не менять между NEW и RESUME.
REPO_ROOT = '/content/semgraphex'

DRIVE_ROOT = '/content/drive/MyDrive/SemanticMap/colab/wishart'
RUN_NAME = 'wishart-grammar-v2-cn100k-l4-bounded-pipeline-01'
BASELINE_RUN_NAME = 'wishart-grammar-v2-cn100k-l4-01'  # optional comparison
RUN_MODE = 'NEW'  # NEW | RESUME
STRICT_RESUME_IDENTITY = True  # не смешивать checkpoints от другого Git revision

# Предпочтительный устойчивый вход. Если файла нет, ноутбук ищет COMPLETED prepared/100k.
DATASET_DRIVE = 'data/conceptnet_en_100k.tsv'
PREPARED_DRIVE = ''  # Например: prepared/prepare-xxxxxxxxxxxx; оставьте '' для AUTO.

REQUIRE_L4 = True
EXPECTED_NODES = 100000
DEVICE = 'cuda'      # Для этого benchmark намеренно strict CUDA.
CPU_WORKERS = 8
BLAS_THREADS = 1  # ОБЯЗАТЕЛЬНО: не ставьте 8 при CPU_WORKERS=8
KEEP_SCRATCH = False

# Один бюджет: pipeline сам делит CPU_WORKERS на extraction threads + VF2 processes.
# Для 8 workers ожидаемый split: 2 + 6. CLI автоматически ограничит budget числом видимых vCPU.

assert RUN_MODE in {'NEW', 'RESUME'}
assert DEVICE == 'cuda', 'Этот notebook предназначен для L4/CUDA benchmark'


In [ ]:
# ===== L4 / RAM / DISK PREFLIGHT =====
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())
disk = shutil.disk_usage('/content')
print('Local /content free GiB:', round(disk.free / 2**30, 2))
try:
    import psutil
    print('RAM total GiB:', round(psutil.virtual_memory().total / 2**30, 2))
except Exception:
    pass

if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU не обнаружен. Выберите Runtime → Change runtime type → L4 GPU.')
gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'
], text=True).strip()
print('GPU:', gpu_line)
if REQUIRE_L4 and 'L4' not in gpu_line.upper():
    raise RuntimeError(f'Ожидался L4 GPU, получено: {gpu_line}')
if disk.free < 8 * 2**30:
    raise RuntimeError('Для полного 100k run требуется не менее 8 GiB свободного /content scratch.')


In [ ]:
# ===== MOUNT GOOGLE DRIVE =====
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE)


In [ ]:
# ===== RESOLVE EXACT 100k INPUT =====
dataset_path = DRIVE / DATASET_DRIVE
prepared_rel = PREPARED_DRIVE.strip()

if prepared_rel:
    prepared_path = DRIVE / prepared_rel
    if not (prepared_path / 'COMPLETED').is_file():
        raise FileNotFoundError(f'Prepared input is not COMPLETED: {prepared_path}')
    INPUT_KIND, INPUT_REL = 'prepared', prepared_rel
elif dataset_path.is_file():
    INPUT_KIND, INPUT_REL = 'dataset', DATASET_DRIVE
else:
    candidates = []
    prepared_root = DRIVE / 'prepared'
    if prepared_root.is_dir():
        candidates = [p for p in prepared_root.iterdir() if p.is_dir() and (p / 'COMPLETED').is_file()]
    # Prefer prepared inputs that are explicitly 100k when names carry the hint;
    # otherwise use the most recently modified completed prepared directory.
    named_100k = [p for p in candidates if '100k' in p.name.lower()]
    pool = named_100k or candidates
    if not pool:
        raise FileNotFoundError(
            f'Нет {dataset_path} и нет COMPLETED prepared input под {prepared_root}. '
            'Поместите ConceptNet 100k TSV в data/conceptnet_en_100k.tsv или задайте PREPARED_DRIVE.'
        )
    chosen = max(pool, key=lambda p: p.stat().st_mtime)
    INPUT_KIND, INPUT_REL = 'prepared', str(chosen.relative_to(DRIVE))

print({'input_kind': INPUT_KIND, 'input_relative': INPUT_REL})


In [ ]:
# ===== CHECKOUT PINNED CODE + REPRODUCIBLE INSTALL =====
repo = Path(REPO_ROOT)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA], check=True)
sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert sha == PINNED_SHA, (sha, PINNED_SHA)
print('Pinned Git SHA:', sha)

constraints = repo / 'requirements/constraints-colab.txt'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', str(constraints), '-e', f'{repo}[dev,wishart,gpu]'
], check=True)

import torch
assert torch.cuda.is_available(), 'PyTorch CUDA unavailable after install'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
if REQUIRE_L4 and 'L4' not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError('PyTorch sees a non-L4 GPU; stop this benchmark.')

from semmap_haken.wishart_hierarchy import _split_full_scan_workers
effective_cpu_workers = min(CPU_WORKERS, max(1, os.cpu_count() or 1))
extract_workers, match_workers = _split_full_scan_workers(effective_cpu_workers)
print('Effective CPU budget:', effective_cpu_workers)
print('Bounded full-scan split:', {'extract_workers': extract_workers, 'match_workers': match_workers, 'sum': extract_workers + match_workers})
assert extract_workers + match_workers == effective_cpu_workers


In [ ]:
# ===== RESUME IDENTITY PREFLIGHT =====
# Проверяем существующий Drive run ДО запуска CLI и даём понятную ошибку.
run_dir = DRIVE / 'runs' / RUN_NAME
manifest_path = run_dir / 'input.json'

if RUN_MODE == 'RESUME':
    if not manifest_path.is_file():
        raise FileNotFoundError(f'RESUME requested but input.json is missing: {manifest_path}')
    previous = json.loads(manifest_path.read_text())
    previous_revision = previous.get('code_revision')
    previous_workers = (previous.get('execution_options') or {}).get('cpu_workers')
    previous_blas = previous.get('blas_threads')
    previous_device = (previous.get('device') or {}).get('selected')
    resume_identity = {
        'run': RUN_NAME,
        'checkpoint_code_revision': previous_revision,
        'notebook_PINNED_SHA': PINNED_SHA,
        'checkpoint_cpu_workers': previous_workers,
        'requested_cpu_workers': CPU_WORKERS,
        'checkpoint_blas_threads': previous_blas,
        'requested_blas_threads': BLAS_THREADS,
        'checkpoint_device': previous_device,
        'requested_device': DEVICE,
    }
    print(json.dumps(resume_identity, indent=2))
    if STRICT_RESUME_IDENTITY and previous_revision != PINNED_SHA:
        raise RuntimeError(
            'Несовместимый RESUME: этот run создан другим Git revision. '
            f'checkpoint={previous_revision}, notebook={PINNED_SHA}. '
            'Чтобы сохранить старый прогресс, продолжайте его старым revision; '
            'для bounded-pipeline используйте новый RUN_NAME и RUN_MODE=NEW.'
        )
else:
    if run_dir.exists() and manifest_path.is_file():
        previous = json.loads(manifest_path.read_text())
        raise RuntimeError(
            f'RUN_MODE=NEW, но run уже существует: {run_dir}. '
            f"Его revision={previous.get('code_revision')}. "
            'Выберите новый RUN_NAME или явно используйте совместимый RESUME.'
        )
    print('NEW run identity is clear:', run_dir)

assert BLAS_THREADS == 1, (
    'Для bounded extraction+VF2 pipeline оставьте BLAS_THREADS=1, '
    'иначе возникает nested CPU oversubscription.'
)


In [ ]:
# ===== BUILD NEW / RESUME COMMAND =====
config = repo / 'configs/wishart_conceptnet_grammar_v2.yaml'
cmd = [
    sys.executable, '-m', 'semmap_haken.wishart_colab_cli',
    '--config', str(config),
    '--drive-root', DRIVE_ROOT,
    '--scratch-root', '/content/semmap-wishart-grammar-v2-pipeline',
    '--run-name', RUN_NAME,
    '--device', DEVICE,
    '--cpu-workers', str(CPU_WORKERS),
    '--blas-threads', str(BLAS_THREADS),
    '--compare-mode', 'size_mtime',
]
cmd += ['--prepared-drive', INPUT_REL] if INPUT_KIND == 'prepared' else ['--dataset-drive', INPUT_REL]
if RUN_MODE == 'RESUME':
    cmd += ['--resume']
if KEEP_SCRATCH:
    cmd += ['--keep-scratch']

print('Command:')
print(' '.join(cmd))
print('Durable run:', DRIVE / 'runs' / RUN_NAME)

print('Expected bounded split:', {'extract_workers': extract_workers, 'match_workers': match_workers})


In [ ]:
# ===== FULL CONCEPTNET-100k / L4 BOUNDED-PIPELINE RUN =====
# В этой ячейке subprocess только оркестрирует CLI. Внутри full census один
# CPU budget делится между extraction threads и VF2 spawn processes.
# При обрыве: RUN_MODE='RESUME' и снова Run all с теми же параметрами.
subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
# ===== EXACTNESS + COMPRESSION + MULTISCALE VALIDATION =====
run = DRIVE / 'runs' / RUN_NAME
assert (run / 'COMPLETED').is_file(), f'Missing COMPLETED: {run}'
input_graph = json.loads((run / 'input_graph_report.json').read_text())
assert int(input_graph['node_count']) == EXPECTED_NODES, input_graph

colab = json.loads((run / 'COLAB_RUN.json').read_text())
hierarchy = json.loads((run / 'hierarchy.json').read_text())
codec = json.loads((run / 'hierarchy_codec_report.json').read_text())
compression = json.loads((run / 'compression_analysis.json').read_text())
multiscale = json.loads((run / 'multiscale_graph_model.json').read_text())

assert codec['roundtrip_exact'] is True
assert compression['summary']['hierarchy_roundtrip_exact'] is True

summary = {
    'run': str(run),
    'git_sha': colab.get('device') and PINNED_SHA,
    'gpu': (colab.get('device') or {}).get('gpu_name'),
    'levels': hierarchy['levels'],
    'nodes': [hierarchy['initial_nodes'], hierarchy['final_nodes']],
    'stop_reason': hierarchy['stop_reason'],
    'dictionary_size': hierarchy.get('dictionary_size'),
    'hierarchy_archive_bytes': compression['summary']['hierarchy_archive_bytes'],
    'binary_bundle_bytes': compression['summary']['hierarchy_binary_bundle_bytes'],
    'hierarchy_ratio_binary': compression['summary']['hierarchy_compression_ratio_binary_bundle'],
    'best_transition_ratio': compression['summary']['best_transition_binary_ratio'],
    'worst_transition_ratio': compression['summary']['worst_transition_binary_ratio'],
    'first_nonwinning_transition': compression['summary']['first_transition_not_beating_binary_baseline'],
    'multiscale_projections': multiscale['projection_count'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nPer-transition bottlenecks:')
for row in compression['transitions']:
    print({
        'level': row['source_level'],
        'nodes': [row['fine_nodes'], row['coarse_nodes']],
        'ratio_binary': row['compression_ratio_binary'],
        'largest_entry': row['largest_entry'],
        'largest_entry_bytes': row['largest_entry_bytes'],
        'timing_seconds': row.get('timing_seconds'),
    })

print('\nГОТОВО:', run)

print('\nFull-scan parallelism and timings:')
parallel_rows = []
for tr in hierarchy.get('transitions', []):
    dm = tr.get('dictionary_metrics') or {}
    timing = tr.get('phase_timing_seconds') or {}
    row = {
        'level': tr.get('source_level'),
        'full_scan_seconds': timing.get('full_scan'),
        'parallelism': dm.get('full_scan_parallelism'),
    }
    parallel_rows.append(row)
    print(row)

baseline_run = DRIVE / 'runs' / BASELINE_RUN_NAME
baseline_hierarchy_path = baseline_run / 'hierarchy.json'
if baseline_hierarchy_path.is_file():
    baseline_hierarchy = json.loads(baseline_hierarchy_path.read_text())
    baseline_by_level = {
        int(tr['source_level']): (
            (tr.get('phase_timing_seconds') or {}).get('full_scan')
        )
        for tr in baseline_hierarchy.get('transitions', [])
        if tr.get('source_level') is not None
    }
    print('\nOptional full_scan speedup vs', BASELINE_RUN_NAME)
    for row in parallel_rows:
        level = row['level']
        new_t = row['full_scan_seconds']
        old_t = baseline_by_level.get(int(level)) if level is not None else None
        if old_t and new_t:
            print({
                'level': level,
                'baseline_seconds': old_t,
                'pipeline_seconds': new_t,
                'speedup_x': old_t / new_t,
            })
else:
    print('\nBaseline run not found; speedup comparison skipped:', baseline_run)


## Продолжение после обрыва

Для **этого bounded-pipeline run** поменяйте только:

```python
RUN_MODE = 'RESUME'
```

Не меняйте `PINNED_SHA`, `RUN_NAME`, вход, `CPU_WORKERS`, `BLAS_THREADS` или `DEVICE`.

### Если у вас уже есть старый run `wishart-grammar-v2-cn100k-l4-01`

Он был создан revision `d70d5133b60d8cd63e8c374501252bf0bc545eac` с `CPU_WORKERS=4`, `BLAS_THREADS=1`. Его нельзя продолжать новым bounded-pipeline revision без смешения экспериментальных условий. Для сохранения его прогресса используйте старый revision и старые параметры. Для нового pipeline используйте отдельный `RUN_NAME='wishart-grammar-v2-cn100k-l4-bounded-pipeline-01'` и начните с `RUN_MODE='NEW'`.

## Что сравнивать

Главный performance-показатель нового pipeline — `phase_timing_seconds.full_scan`. `dictionary_metrics.full_scan_parallelism` сохраняет фактический CPU split и bounded queue limits. Если старый run присутствует на Drive, финальная ячейка сравнит timings доступных завершённых уровней.
